In [8]:
import pandas as pd

df = pd.read_excel("../data/online_retail.xlsx")
df.to_csv("../data/online_retail.csv", index=False)


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os



In [10]:
import pandas as pd
import os

DATA_PATH = os.path.join("..", "data", "online_retail.csv")
retail_data = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")


In [11]:
print("Shape:", retail_data.shape)
retail_data.info()


Shape: (541909, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [12]:
retail_data.columns


Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

In [13]:
retail_data.isna().sum().sort_values(ascending=False).head(10)


CustomerID     135080
Description      1454
InvoiceNo           0
StockCode           0
Quantity            0
InvoiceDate         0
UnitPrice           0
Country             0
dtype: int64

In [14]:
# Check canceled invoices
canceled = retail_data[retail_data['InvoiceNo'].astype(str).str.startswith('C')]

print("Canceled invoices:", canceled.shape[0])
canceled.head()


Canceled invoices: 9288


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


### Data Cleaning Steps

- Removed canceled transactions (InvoiceNo starting with 'C')
- Removed rows with missing descriptions
- Removed rows with missing or blank StockCode (used as Product ID)
- Removed non-positive quantities
- Removed duplicate products within the same invoice


In [15]:
# Remove canceled invoices
retail_clean = retail_data[
    ~retail_data['InvoiceNo'].astype(str).str.startswith('C')
].copy()

print("After removing canceled invoices:", retail_clean.shape)


After removing canceled invoices: (532621, 8)


In [16]:
# Remove rows with missing Description
retail_clean = retail_clean.dropna(subset=['Description'])

# Convert Description to string (safety)
retail_clean['Description'] = retail_clean['Description'].astype(str)

print("After removing null descriptions:", retail_clean.shape)


After removing null descriptions: (531167, 8)


In [17]:
retail_clean = retail_clean.dropna(subset=["StockCode"])


In [18]:
retail_clean = retail_clean[retail_clean["StockCode"].astype(str).str.strip() != ""]


In [19]:
# Keep only positive quantities
retail_clean = retail_clean[retail_clean['Quantity'] > 0]

print("After removing non-positive quantities:", retail_clean.shape)


After removing non-positive quantities: (530693, 8)


In [20]:
retail_clean.info()


<class 'pandas.core.frame.DataFrame'>
Index: 530693 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    530693 non-null  object 
 1   StockCode    530693 non-null  object 
 2   Description  530693 non-null  object 
 3   Quantity     530693 non-null  int64  
 4   InvoiceDate  530693 non-null  object 
 5   UnitPrice    530693 non-null  float64
 6   CustomerID   397924 non-null  float64
 7   Country      530693 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 36.4+ MB


In [21]:
# 2) Remove non-product / special StockCodes (Discount, postage, bank charges, manual adjustments etc.)
invalid_stockcodes = [
    "D", "POST", "BANK CHARGES", "C2", "M", "DOT", "CRUK"
]

retail_clean["StockCode"] = retail_clean["StockCode"].astype(str).str.strip()

retail_clean = retail_clean[~retail_clean["StockCode"].isin(invalid_stockcodes)]

print("After removing invalid StockCodes:", retail_clean.shape)


After removing invalid StockCodes: (528379, 8)


In [22]:
retail_clean["Description"] = retail_clean["Description"].str.strip()
retail_clean = retail_clean[~retail_clean["Description"].str.lower().isin([
    "test", "returned", "taig adjust", "website fixed"
])]


In [23]:
# Remove duplicate items inside the same invoice (keep presence/absence)
before = retail_clean.shape[0]

retail_clean = retail_clean.drop_duplicates(subset=["InvoiceNo", "StockCode"])

after = retail_clean.shape[0]
print("Rows before:", before)
print("Rows after :", after)
print("Removed    :", before - after)


Rows before: 528372
Rows after : 517871
Removed    : 10501


In [24]:
# ---------- Build baskets using StockCode (FINAL & CORRECT) ----------

# Remove duplicate item per invoice (presence/absence)
retail_clean = retail_clean.drop_duplicates(subset=["InvoiceNo", "StockCode"])

# Group into baskets
basket_df = (
    retail_clean
    .groupby("InvoiceNo")["StockCode"]
    .apply(list)
    .reset_index(name="items")
)

print("Total transactions:", basket_df.shape[0])

transactions = basket_df["items"].tolist()
transactions[:3]



Total transactions: 19955


[['85123A', '71053', '84406B', '84029G', '84029E', '22752', '21730'],
 ['22633', '22632'],
 ['84879',
  '22745',
  '22748',
  '22749',
  '22310',
  '84969',
  '22623',
  '22622',
  '21754',
  '21755',
  '21777',
  '48187']]

In [25]:
!pip install -q mlxtend


In [26]:
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)

basket_encoded = pd.DataFrame(te_ary, columns=te.columns_)
print(basket_encoded.shape)

basket_encoded.head()


(19955, 3920)


,10002,10080,10120,10123C,10124A,10124G,10125,10133,10135,11001,...,DCGSSBOY,DCGSSGIRL,PADS,S,gift_0001_10,gift_0001_20,gift_0001_30,gift_0001_40,gift_0001_50,m
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [27]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(
    basket_encoded,
    min_support=0.02,   # you can change later
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False)
frequent_itemsets.head(10)


,support,itemsets
291,0.110398,(85123A)
288,0.104836,(85099B)
121,0.099674,(22423)
247,0.084490,(47566)
12,0.078426,(20725)
277,0.072914,(84879)
103,0.069757,(22197)
170,0.069406,(22720)
33,0.066149,(21212)
114,0.064395,(22383)


In [28]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1
)

rules = rules.sort_values(["lift", "confidence", "support"], ascending=False)
rules.head(10)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
44,(22698),"(22697, 22699)",0.038386,0.038487,0.027161,0.707572,18.384890,1.0,0.025684,3.288032,0.983355,0.546371,0.695867,0.706650
41,"(22697, 22699)",(22698),0.038487,0.038386,0.027161,0.705729,18.384890,1.0,0.025684,3.267784,0.983457,0.546371,0.693982,0.706650
42,"(22698, 22699)",(22697),0.030018,0.050864,0.027161,0.904841,17.789271,1.0,0.025634,9.974249,0.972993,0.505597,0.899742,0.719416
43,(22697),"(22698, 22699)",0.050864,0.030018,0.027161,0.533990,17.789271,1.0,0.025634,2.081463,0.994364,0.505597,0.519569,0.719416
14,(22697),(22698),0.050864,0.038386,0.031721,0.623645,16.246531,1.0,0.029769,2.555073,0.988740,0.551394,0.608622,0.725008
15,(22698),(22697),0.038386,0.050864,0.031721,0.826371,16.246531,1.0,0.029769,5.466450,0.975910,0.551394,0.817066,0.725008
40,"(22697, 22698)",(22699),0.031721,0.053420,0.027161,0.856240,16.028397,1.0,0.025467,6.584451,0.968327,0.468453,0.848127,0.682341
45,(22699),"(22697, 22698)",0.053420,0.031721,0.027161,0.508443,16.028397,1.0,0.025467,1.969819,0.990525,0.468453,0.492339,0.682341
35,(23300),(23301),0.037985,0.045753,0.027362,0.720317,15.743612,1.0,0.025624,3.411883,0.973459,0.485333,0.706907,0.659173
34,(23301),(23300),0.045753,0.037985,0.027362,0.598028,15.743612,1.0,0.025624,2.393241,0.981383,0.485333,0.582156,0.659173


In [29]:
# --- Proper deduplication + re-sorting + top 10 ---

# Create readable rule string (order-independent)
rules = rules.copy()
rules["rule"] = (
    rules["antecedents"].apply(lambda x: ", ".join(sorted(list(x))))
    + " -> " +
    rules["consequents"].apply(lambda x: ", ".join(sorted(list(x))))
)

# Drop duplicate rules
rules_dedup = rules.drop_duplicates(subset=["rule"]).copy()

# Re-sort AFTER deduplication (very important)
rules_dedup = rules_dedup.sort_values(
    ["lift", "confidence", "support"],
    ascending=False
)

# Final top 10
top10_rules = rules_dedup[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].head(10).copy()

# Make readable
top10_rules["antecedents"] = top10_rules["antecedents"].apply(lambda x: ", ".join(sorted(list(x))))
top10_rules["consequents"] = top10_rules["consequents"].apply(lambda x: ", ".join(sorted(list(x))))

top10_rules


,antecedents,consequents,support,confidence,lift
44,22698,"22697, 22699",0.027161,0.707572,18.384890
41,"22697, 22699",22698,0.027161,0.705729,18.384890
42,"22698, 22699",22697,0.027161,0.904841,17.789271
43,22697,"22698, 22699",0.027161,0.533990,17.789271
14,22697,22698,0.031721,0.623645,16.246531
15,22698,22697,0.031721,0.826371,16.246531
40,"22697, 22698",22699,0.027161,0.856240,16.028397
45,22699,"22697, 22698",0.027161,0.508443,16.028397
35,23300,23301,0.027362,0.720317,15.743612
34,23301,23300,0.027362,0.598028,15.743612


In [30]:
# --- STEP 4: Map StockCode -> Description (FINAL OUTPUT ONLY) ---

# 1) Create mapping once
code_to_desc = (
    retail_clean
    .drop_duplicates("StockCode")
    .set_index("StockCode")["Description"]
    .to_dict()
)

# 2) Apply mapping ONLY to final Top-10 table
top10_rules_readable = top10_rules.copy()

top10_rules_readable["antecedents"] = top10_rules_readable["antecedents"].apply(
    lambda s: ", ".join(code_to_desc.get(x, x) for x in s.split(", "))
)

top10_rules_readable["consequents"] = top10_rules_readable["consequents"].apply(
    lambda s: ", ".join(code_to_desc.get(x, x) for x in s.split(", "))
)

top10_rules_readable


,antecedents,consequents,support,confidence,lift
44,PINK REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...",0.027161,0.707572,18.384890
41,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...",PINK REGENCY TEACUP AND SAUCER,0.027161,0.705729,18.384890
42,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...",GREEN REGENCY TEACUP AND SAUCER,0.027161,0.904841,17.789271
43,GREEN REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...",0.027161,0.533990,17.789271
14,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.031721,0.623645,16.246531
15,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.031721,0.826371,16.246531
40,"GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY ...",ROSES REGENCY TEACUP AND SAUCER,0.027161,0.856240,16.028397
45,ROSES REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY ...",0.027161,0.508443,16.028397
35,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,0.027362,0.720317,15.743612
34,GARDENERS KNEELING PAD KEEP CALM,GARDENERS KNEELING PAD CUP OF TEA,0.027362,0.598028,15.743612


In [32]:
import os

os.makedirs("../outputs", exist_ok=True)

save_path = os.path.join("..", "outputs", "top10_rules.csv")
top10_rules_readable.to_csv(save_path, index=False)

print("Saved:", os.path.abspath(save_path))


Saved: c:\Users\janvi\OneDrive\Desktop\projects\association-rule-mining-retail\outputs\top10_rules.csv
